# Procedure 与人工判断

先会运行参数与扫描课程，再选本课。目标：把合成采集和人工判断放进一个可恢复的 procedure；查看原问题后，以类型化答案回答，最后显式继续。

无需设备，也不需要前课数据、候选发布或后台任务调度。Help 已准备普通作者目录；在外部编辑器选择本目录 `.venv` 内核，逐格运行。源码是可编辑的 `src/my_experiment/procedure.py`。

完成标准：看到 waiting_for_input，检查曲线并记录一次判断，再显式 resume，看到 succeeded。完成表示流程结束，不等于答案正确或发布了标定。

In [ ]:
import scopecat as sc

session = sc.notebook()
session

## 重开时先找历史
重开或服务重启后，先运行连接和历史格，再用 `task = session.procedures.get("原 ID")` 找回原任务。若仍在 waiting_for_input，直接运行“先查看，再回答”及后续格，不必准备新请求。不要用 Run All 恢复；下面“新采集”会提交新任务。

In [ ]:
from my_experiment.lesson_identity import IDENTITY

cursor = None
while True:
    page = session.procedures.list(cursor=cursor)
    for item in page.items:
        if (item.request_key or "").startswith(IDENTITY + "-"):
            print(item.procedure_run_id, item.state)
    cursor = page.next_cursor
    if cursor is None:
        break

## 准备本课输入
打开可编辑的 `src/my_experiment/procedure.py`：`survey` 顺序写出扫描、曲线、`choose_peak` 人工判断。后者是同文件中的普通 Python 函数，继续使用 `ctx.interpret` 保存问题和答案；不是新的框架 API。重放使用这些步骤的已存结果，不再次扫描。下一格只准备输入。

In [ ]:
from uuid import uuid4

from my_experiment.procedure import SurveyIntent, survey
from my_experiment.setup import open_parameters

params = open_parameters(session)
saved = params.save(note="人工判断课程输入")
setup = session.setup.get(f"{IDENTITY}-setup")
intent = SurveyIntent(
    config=session.parameters.resolve(saved, setup=setup), setup=setup.ref
)
prepared = session.procedures.prepare(
    survey, intent, request_key=IDENTITY + "-" + uuid4().hex[:8]
)

## 新采集：显式提交一次
下一格启动一个新任务。wait 只观察到完成、人工输入或需处理状态，不自动回答或继续。保存打印的 ID。

In [ ]:
task = prepared.submit()
print("保留的任务 ID:", task.id)
print(task.wait(timeout=120).state)

## 先查看，再回答
省略步骤 key 会读取唯一当前待答问题，包括修正回答后的下一轮。取得 pending 保存本次实际查看的问题和证据。若题目变了，旧对象不能回答新题；冲突后重新取得、重新查看，不自动重发。

曲线是向上的峰，不是谷。找响应最高点，读取横轴频率（MHz）；应用 Decisions 中可悬停读数，但点击曲线不会填入答案。下面单独的打开单元格显示应用中同一份保存的曲线，不重新采集。看完后返回 Notebook 回答。

In [ ]:
from my_experiment.procedure import PEAK_CHOICE, PeakChoice

pending = task.pending_input(schema=PEAK_CHOICE)
print(pending.request.title)
print(pending.request.instructions)
print(pending.inputs)

需要在桌面查看时，运行下一格。它打开同一应用的既有对象，不采集；请先保持应用桌面运行。若未确认打开，先检查已有窗口，再用异常的 `request.wait()` 重试同一请求，避免重新打开。

In [ ]:
task.open(view="decision")

## 记录判断（不继续执行）
检查曲线后编辑下一格：选峰填 6490–6510 MHz 内的频率；不确定用 `PeakChoice("Unsure", None)`，没有明确峰用 `PeakChoice("No clear peak", None)`。示例的峰约 6500 MHz。

也可在应用 Decisions 填写；两条入口择一回答同一问题，另一条入口不要再提交。GUI 的输入提示不代替 procedure 的科学校验。

In [ ]:
answer = PeakChoice("Peak selected", 6500.0)
receipt = pending.respond(answer, actor="learner")
print(receipt)
print(task.snapshot.state)  # ready：已记录，尚未继续

## 显式继续并查看结果
确认后续操作合适，再运行下一格。本课只检查答案组合并结束，不发布参数。若填写越界或组合不一致，会出现新问题及具体原因：回到“先查看，再回答”重新取得 pending，再记录并继续；旧扫描保留。

普通应用 worker 在人工输入处暂停。若另有独立驻留执行器，先确认它的继续策略，不能把 respond 当成额外暂停保障。

In [ ]:
task.resume()
result = task.wait(timeout=120)
print(result.state, result.closure)
if result.state == "waiting_for_input":
    print("需要修正判断；回到查看问题的单元格。")

## 完成后改写
`succeeded` 表示答案满足本课的范围与组合约束，不是自动判分。在同一份作者源码中，`peak_choice_problem` 集中列出范围、必填和留空规则；`choose_peak` 展示问题并在必要时重问，返回包含 `.value` 和 `.ref` 的完整判断。科学判断仍由你作出，后续步骤可用 `.ref` 保留依据。

`round_index` 表示新的判断轮次（`choose-1`、`choose-2`），不是某一步副作用的重试 attempt。不要为恢复原任务而重命名这些 key；也不要把新扫描放进重问循环。

修改合成 response、扫描范围或问题时，同步修改校验，升级 procedure version，保存、显式 `session.refresh()` 并重新导入后准备新请求。旧任务仍绑定旧源码。继续学习自动验证与参数发布时，再选 calibration；本课无需高级调度。